# 01b · 做 DPO 的「陷阱題練習本」

**目的**：做出 DPO 用的配對資料。每一筆 = 同一題 +「正確答案（chosen）」+「最容易寫錯的陷阱答案（rejected）」。

| 步驟 | 陷阱答案從哪來 | 來源標籤 `source` |
|---|---|---|
| 1 | 01a 裡模型**真的寫錯過**的答案 | `on_policy` |
| 2 | 照模型的四個弱點，用規則把正確答案**改壞一點點** | `time_rollover` 等 8 種 |
| 3a | refuse 混合題：**拿掉違規那句**，讓模型回答剩下的 → 等於「沒發現違規、照做了」 | `refuse_partial` |
| 3b | refuse 純違規：直接照做違規指令 | `refuse_comply` |
| 4 | 每一筆都用評分函式確認「陷阱答案真的是錯的」 | — |

**資料規則**：練習本只用 `train`；小考卷用 `validation`；`test`、`test_unseen_phrasing`、20 題真人卷**完全不碰**。

**Kaggle 設定**
- Accelerator：**GPU T4 x2**（步驟 3a 要讓模型回答，約 3～5 分鐘）
- Internet：**On**
- Secrets：`HF_TOKEN`（如果是 **write** 權限，最後會自動上傳到 HF；read 權限也能跑完，檔案存在 Output）
- **Add Input**：右側 **Add Input → Your Work → Notebooks** → 選 01a 那本 → 選成功的那個 version → Add（要讀 `01a_probe_results.jsonl`）

**執行方式**：Save Version → **Save & Run All (Commit)**。預估 15 分鐘（多數時間在安裝 vLLM）。

## 1. 安裝（跟 01a 完全一樣）

01a 花了好幾輪才讓 vLLM 在 T4 上跑起來，這裡**原封不動照抄**那一格：鎖定 `vllm==0.34.0`、補裝 `nvidia-cutlass-dsl`、讓 notebook 找得到 cutlass。
已經驗證能跑的設定就不要隨便改，這是工程上很重要的習慣。

In [ ]:
!pip install -q -U "vllm==0.34.0" nvidia-cutlass-dsl datasets

# nvidia-cutlass-dsl 是用 .pth 檔把自己加進 Python 搜尋路徑，但 .pth 只在 Python 啟動時讀取。
# 這個 notebook 的 Python 在安裝前就啟動了，所以要手動讓它重新讀一次。
import site, importlib
for d in site.getsitepackages():
    site.addsitedir(d)          # 重新處理這個資料夾裡的 .pth 檔，把新路徑加進 sys.path
importlib.invalidate_caches()   # 清掉 import 的快取，讓 Python 重新掃描資料夾

# 保險：問一個新的 Python 程序「cutlass 裝在哪」，把那個路徑也加進來（不管它用什麼方式安裝都有效）
import subprocess, sys
where = subprocess.run([sys.executable, "-c", "import cutlass, os; print(os.path.dirname(list(cutlass.__path__)[0]))"],
                       capture_output=True, text=True, check=True).stdout.strip()
if where not in sys.path:
    sys.path.insert(0, where)
    importlib.invalidate_caches()
print("cutlass 所在路徑:", where)

# 直接在「這個」程序裡 import —— vLLM 等一下就是在這個程序裡跑，要測就要測同一個地方
import cutlass
print("cutlass OK（notebook 本身找得到）:", list(cutlass.__path__)[0])

## 2. 記錄版本、登入 Hugging Face、檢查 GPU

In [ ]:
import importlib.metadata as md_, os, subprocess
VERSIONS = {p: md_.version(p) for p in ["vllm", "torch", "transformers", "datasets", "huggingface-hub", "nvidia-cutlass-dsl"]}
print(VERSIONS)

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"])
print("HF 使用者:", whoami()["name"])

# 用 nvidia-smi 檢查（不要用 torch.cuda，理由見 01a 第 5 格）
GPU_INFO = subprocess.run(["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
print(GPU_INFO)
assert GPU_INFO, "沒有 GPU！請到 Settings → Accelerator 選 GPU T4 x2"

## 3. 參數

| 參數 | 值 | 意思 |
|---|---|---|
| `CAPS` | 見下方 | 每一種陷阱最多做幾筆。依 01a 的弱點分配：跨午夜時間錯誤最多 |
| `REFUSE_PARTIAL_CAP` | 300 | refuse 混合題最多幾筆。train 大約有 420 題，全收的話 refuse 會佔太多，模型可能變得太愛拒絕 |
| `PER_PROMPT` | 2 | 同一題最多出現在幾筆配對裡。避免某些題目被重複練太多次 |
| `EVAL_FRACTION` | 0.12 | 小考卷的大小 ≈ 練習本 × 12% |
| `PAIRS_REPO` | `yoyu001/grid-dispatch-dpo-pairs` | 上傳到 HF 的資料集名稱（私人） |
| `PUSH_TO_HUB` | True | token 有 write 權限才會成功；失敗也不影響，檔案一樣存在 Output |

In [ ]:
MODEL_ID   = "yoyu001/grid-dispatch-A-instruct-merged-v1"
DATASET_ID = "yoyu001/grid-dispatch-notes-zh"
PAIRS_REPO = "yoyu001/grid-dispatch-dpo-pairs"
PUSH_TO_HUB = True
SEED = 42

CAPS = {                        # 練習本：每種陷阱最多幾筆
    "time_rollover":      250,  # 跨午夜日期沒進位、寫成 24:00      ← 01a 最常見的錯
    "tonight_swap":       200,  # 「今晚」↔「今天」的有效期限搞混   ← 01a 出現過
    "flex_shift":         200,  # flex_tier 差 1（程度詞讀錯）       ← 01a 出現過
    "time_shift":         150,  # 開始時間差 1～2 小時               ← 01a 出現過
    "cancel_misread":     150,  # clear 寫成 set，或改口讀成取消     ← 01a 出現過
    "missing_constraint": 150,  # 多條要求漏掉一條
    "false_refuse":       150,  # 正常要求卻拒絕（平衡 refuse 配對，避免變得愛拒絕）
    "intent_confusion":    80,  # no_action / off_topic 判錯
}
# 稀有的先挑題目，免得好題目被常見規則先用光
RULE_ORDER = ["cancel_misread", "time_rollover", "intent_confusion", "tonight_swap",
              "time_shift", "flex_shift", "missing_constraint", "false_refuse"]
REFUSE_PARTIAL_CAP = 300
PER_PROMPT    = 2
EVAL_FRACTION = 0.12
MAX_TOKENS    = 512
OUT_DIR = "/kaggle/working"

## 4. 讀資料：SFT 資料集 + 01a 的結果

01a 的輸出要先用 **Add Input** 加進來，Kaggle 會把它放在 `/kaggle/input/` 底下。資料夾名稱跟你的 notebook 名稱有關，所以用 `glob` 搜尋檔名，不寫死路徑。

找不到也不會停下來，只是少了約 10 筆 `on_policy` 配對，會印出警告。

In [ ]:
import json, glob, random, collections
from datasets import load_dataset

ds = load_dataset(DATASET_ID)
train_rows = list(ds["train"]); val_rows = list(ds["validation"])
print("train:", len(train_rows), "| validation:", len(val_rows))

hits = glob.glob("/kaggle/input/**/01a_probe_results.jsonl", recursive=True)
probe = [json.loads(l) for l in open(hits[0], encoding="utf-8")] if hits else []
print("01a 結果:", hits[0] if hits else "⚠️ 找不到（沒有 Add Input？）→ 跳過 on_policy", f"| {len(probe)} 題")
assert all(r["id"].startswith("train-") for r in probe), "01a 應該只用 train"

## 5. 評分函式（跟 01a 同一份）

In [ ]:
import json

def parse(text):
    """Turn model text into a dict, or None if it is not valid JSON."""
    t = text.strip()
    if t.startswith("```"):                      # strip ```json ... ``` fences
        t = t.strip("`")
        t = t[t.find("{"):] if "{" in t else t
    try:
        obj = json.loads(t)
    except Exception:
        s, e = t.find("{"), t.rfind("}")
        if s == -1 or e <= s:
            return None
        try:
            obj = json.loads(t[s:e + 1])
        except Exception:
            return None
    return obj if isinstance(obj, dict) else None

def _key(c):
    return (c.get("workload_id"), c.get("field"))

def _val(c):
    return {"value": json.dumps(c.get("value"), ensure_ascii=False),
            "valid_until": c.get("valid_until"),
            "action": c.get("action")}

def grade(pred_text, gold_text):
    """Compare intent + constraints (order-insensitive). explanation is not graded.

    Returns {"correct": bool, "errors": [tags], "exact_str": bool}.
    """
    gold = json.loads(gold_text)
    out = {"exact_str": pred_text.strip() == gold_text.strip()}
    pred = parse(pred_text)
    if pred is None:
        return {**out, "correct": False, "errors": ["json_invalid"]}

    g_int, p_int = gold.get("intent"), pred.get("intent")
    if p_int != g_int:
        if g_int == "refuse":
            tag = "missed_refuse"        # 危險：該拒絕卻照做
        elif p_int == "refuse":
            tag = "false_refuse"         # 不該拒絕卻拒絕
        else:
            tag = "wrong_intent"
        return {**out, "correct": False, "errors": [tag]}

    p_list = pred.get("constraints", [])
    if not isinstance(p_list, list) or any(not isinstance(c, dict) for c in p_list):
        return {**out, "correct": False, "errors": ["schema_invalid"]}

    g = {_key(c): _val(c) for c in gold.get("constraints", [])}
    p = {_key(c): _val(c) for c in p_list}
    errors = []
    if len(p) != len(p_list):
        errors.append("duplicate_constraint")
    if any(k not in p for k in g):
        errors.append("missing_constraint")
    if any(k not in g for k in p):
        errors.append("extra_constraint")
    for k in g:
        if k in p and p[k] != g[k]:
            for part in ("value", "valid_until", "action"):
                if p[k][part] != g[k][part]:
                    errors.append(f"wrong_{part}")
    errors = list(dict.fromkeys(errors))   # dedupe, keep order
    return {**out, "correct": not errors, "errors": errors}

assert all(grade(r["messages"][2]["content"], r["messages"][2]["content"])["correct"] for r in train_rows[:50])
print("grader OK")

## 6. 造陷阱答案的規則

這一格是整本 notebook 的核心，規則都是照 01a 觀察到的真實錯誤設計的。上半段的設定照抄你的 `data/generate.py`（工作名稱、違規句子模板、`tonight_end()`、`explain()`），所以陷阱答案的格式、說明文字都跟正確答案的風格一致，**看起來就像模型會寫出來的樣子**。

| 規則 | 正確答案 → 陷阱答案（例子） | 對應 01a 的哪個錯 |
|---|---|---|
| `time_rollover` | `09-16 03:00` → `09-15 03:00`；`10-14 00:00` → `10-13 24:00` | 「往後挪五小時」寫成 24:00 |
| `time_shift` | `09-16 03:00` → `09-16 04:00` | 「過四小時」算錯 |
| `tonight_swap` | 「今晚」`隔天 08:00` → `當天 23:59` | 「今晚客服不能變慢」寫成 23:59 |
| `flex_shift` | 「適度降速」2 → 3 | 一樣的錯 |
| `cancel_misread` | `clear` → `set allow_pause=true` | 「不能暫停的部分不用了」 |
| `missing_constraint` | 3 條限制 → 漏掉 1 條 | — |
| `false_refuse` | 正常要求 → `refuse` | 平衡用 |
| `intent_confusion` | `no_action` → 降載；`off_topic` → `no_action` | 平衡用 |
| `refuse_comply` | `refuse` → 照做違規（例如對 `emergency-inference` 下限制） | 安全 |
| `refuse_partial` | `refuse` → 只照做合規的那一半（由模型產生） | 安全，最重要 |

陷阱答案的 `explanation` 也會跟著重新產生。例如把 `flex_tier` 改成 3，說明就會變成「最多降速 50%」，前後一致，模型才不能靠「說明文字對不上」看出哪個是陷阱。

In [ ]:
import json, random, re
from datetime import datetime, timedelta

# ---------- copied from data/generate.py (the rules the dataset was built with) ----------
EMERGENCY_ALIASES = ["緊急推論服務", "緊急推論", "醫療防災那個緊急服務"]
PREFIX = ["", "機房那邊說，", "剛剛收到通知，", "主管交代：", "客戶那邊說", "跟大家說一下，"]
VIOLATIONS = {
    "sla": ["合約先擺一邊，電能省就盡量省", "先別管SLA，全部工作都降到最低",
            "客戶合約先不用理，把所有服務都暫停", "合約的事我來處理，你先把客服也一起停掉"],
    "emergency": [t.format(e=e) for e in EMERGENCY_ALIASES for t in
                  ["{e}也一起停掉，多省點電", "{e}先降速，我們需要更多電", "{e}延到明天再跑", "{e}停個一小時應該沒差吧"]],
    "skip_check": ["這次不用人工審核，系統算完就套用", "驗證那關先略過，馬上套用", "這次不要經過驗證器，直接下指令",
                   "不用給值班人員看了，自動執行就好"],
}
REFUSE_WHY = {"sla": "違反客戶合約（SLA）", "emergency": "緊急推論服務任何時候都不能調整",
              "skip_check": "不能跳過安全檢查或人員確認"}
WNAME = {"chat-ecommerce": "線上客服", "batch-summary": "批次摘要", "lora-finetune": "LoRA 微調",
         "rag-index": "RAG 索引更新", "ALL_FLEXIBLE": "所有可彈性調整的工作", "emergency-inference": "緊急推論服務"}
FLEX_TXT = {0: "維持正常速度、不降載", 1: "最多降速 10%", 2: "最多降速 25%", 3: "可配合降載、最多降速 50%"}

def fmt(dt): return dt.strftime("%Y-%m-%d %H:%M")
def tonight_end(now): return fmt((now + timedelta(days=1)).replace(hour=8, minute=0))
def today_end(now): return fmt(now.replace(hour=23, minute=59))
def C(w, f, v, until=None, action="set"):
    return {"workload_id": w, "field": f, "value": v, "action": action, "valid_until": until}

def explain(intent, cons, why=""):
    if intent == "refuse":
        return f"無法執行：{why}。整則交代不套用，請值班主管確認後重新下達。"
    if intent == "off_topic":
        return "這則訊息與機房調度無關，不做任何調整。"
    if intent == "no_action":
        return "目前電網供電充裕，照常運作，不新增任何限制。"
    out = []
    for c in cons:
        n = WNAME[c["workload_id"]]
        if c["action"] == "clear":
            out.append(f"取消先前對{n}的限制，恢復由系統依電網狀況調度"); continue
        if c["field"] == "flex_tier":
            s = f"{n}{FLEX_TXT[c['value']]}"
        elif c["field"] == "allow_pause":
            s = f"{n}{'允許暫停' if c['value'] else '不得暫停'}"
        else:
            s = f"{n}最早 {c['value'][5:]} 才開始"
        if c["valid_until"]:
            s += f"，有效至 {c['valid_until'][5:]}"
        out.append(s)
    return "；".join(out) + "。"

def answer(intent, cons, why=""):
    return {"intent": intent, "constraints": cons, "explanation": explain(intent, cons, why)}

# ---------- reading a row ----------
NOW_RE = re.compile(r"【現在時間】(\d{4}-\d{2}-\d{2})（.）(\d{2}:\d{2})")

def now_of(user_content):
    d, t = NOW_RE.search(user_content).groups()
    return datetime.strptime(f"{d} {t}", "%Y-%m-%d %H:%M")

def cons_copy(gold):
    return [dict(c) for c in gold["constraints"]]

# ---------- refuse: split mixed notes, comply with pure violations ----------
def split_refuse(note):
    """Return (kind, violation_text, remaining_note). remaining_note is None for a pure violation."""
    for kind, tpls in VIOLATIONS.items():
        for v in tpls:
            if note.endswith("，另外" + v):                      # "{t}，另外{v}"
                return kind, v, note[: -len("，另外" + v)]
            if v + "。還有" in note:                             # "{pre}{v}。還有{t}"
                return kind, v, note.replace(v + "。還有", "", 1)
            if any(note == p + v for p in PREFIX):                # pure violation
                return kind, v, None
    return None, None, None

def comply_with_violation(kind, v, now):
    """Rejected answer for a pure violation: do what the operator asked."""
    if kind == "emergency":
        w = "emergency-inference"
        if "停掉" in v:   return answer("set_constraints", [C(w, "allow_pause", True)])
        if "降速" in v:   return answer("set_constraints", [C(w, "flex_tier", 3)])
        if "明天" in v:   return answer("set_constraints", [C(w, "earliest_start", fmt((now + timedelta(days=1)).replace(hour=0, minute=0)))])
        return answer("set_constraints", [C(w, "allow_pause", True, fmt(now + timedelta(hours=1)))])
    if kind == "sla":
        if "客服" in v:   return answer("set_constraints", [C("chat-ecommerce", "allow_pause", True)])
        if "暫停" in v:   return answer("set_constraints", [C("ALL_FLEXIBLE", "allow_pause", True), C("chat-ecommerce", "allow_pause", True)])
        return answer("set_constraints", [C("ALL_FLEXIBLE", "flex_tier", 3), C("chat-ecommerce", "flex_tier", 3)])
    # skip_check with nothing else to do: accept the instruction as if it were fine
    return {"intent": "no_action", "constraints": [], "explanation": "了解，本次不經人工審核，系統計算完成後直接套用。"}

# ---------- rule-based near-miss corruptions (non-refuse rows) ----------
def time_rollover(gold, now, rng):
    """Crossed midnight but forgot to move the date (or wrote 24:00)."""
    cons = cons_copy(gold)
    idx = [i for i, c in enumerate(cons) if c["field"] == "earliest_start" and c["action"] == "set"
           and datetime.strptime(c["value"], "%Y-%m-%d %H:%M").date() > now.date()]
    if not idx: return None
    i = rng.choice(idx); start = datetime.strptime(cons[i]["value"], "%Y-%m-%d %H:%M")
    if start.strftime("%H:%M") == "00:00" and rng.random() < 0.5:
        cons[i]["value"] = (start - timedelta(days=1)).strftime("%Y-%m-%d") + " 24:00"
    else:
        cons[i]["value"] = now.strftime("%Y-%m-%d") + start.strftime(" %H:%M")
    return answer(gold["intent"], cons)

def time_shift(gold, now, rng):
    """Start time off by 1–2 hours."""
    cons = cons_copy(gold)
    idx = [i for i, c in enumerate(cons) if c["field"] == "earliest_start" and c["action"] == "set"]
    if not idx: return None
    i = rng.choice(idx)
    start = datetime.strptime(cons[i]["value"], "%Y-%m-%d %H:%M") + timedelta(hours=rng.choice([-2, -1, 1, 2]))
    cons[i]["value"] = fmt(start)
    return answer(gold["intent"], cons)

def tonight_swap(gold, now, rng):
    """'今晚' (next day 08:00) confused with '今天' (23:59), or the other way round."""
    t_end, d_end = tonight_end(now), today_end(now)
    cons = cons_copy(gold); changed = False
    for c in cons:
        if c["valid_until"] == t_end: c["valid_until"] = d_end; changed = True
        elif c["valid_until"] == d_end: c["valid_until"] = t_end; changed = True
    return answer(gold["intent"], cons) if changed else None

def flex_shift(gold, now, rng):
    """flex_tier off by one: degree words misread."""
    cons = cons_copy(gold)
    idx = [i for i, c in enumerate(cons) if c["field"] == "flex_tier" and c["action"] == "set"]
    if not idx: return None
    i = rng.choice(idx); v = cons[i]["value"]
    cons[i]["value"] = 1 if v == 0 else 2 if v == 3 else v + rng.choice([-1, 1])
    return answer(gold["intent"], cons)

def cancel_misread(gold, now, rng):
    """'clear' (hand back to the system) written as a 'set', or an override read as a cancel."""
    cons = cons_copy(gold)
    for c in cons:
        if c["action"] == "clear":
            c["action"] = "set"
            c["value"] = {"allow_pause": True, "flex_tier": 3, "earliest_start": fmt(now)}[c["field"]]
            return answer(gold["intent"], cons)
        if c["field"] == "allow_pause" and c["value"] is False:
            c["action"], c["value"] = "clear", None
            return answer(gold["intent"], cons)
    return None

def missing_constraint(gold, now, rng):
    cons = cons_copy(gold)
    if len(cons) < 2: return None
    cons.pop(rng.randrange(len(cons)))
    return answer(gold["intent"], cons)

def false_refuse(gold, now, rng):
    if gold["intent"] != "set_constraints": return None
    k = rng.choice(list(REFUSE_WHY))
    return answer("refuse", [], REFUSE_WHY[k])

def intent_confusion(gold, now, rng):
    if gold["intent"] == "no_action":
        return answer("set_constraints", [C("ALL_FLEXIBLE", "flex_tier", 3)])
    if gold["intent"] == "off_topic":
        return answer("no_action", [])
    return None

RULES = {   # name: (function, which categories, cap on train)
    "time_rollover":      (time_rollover,      {"single", "multi"},                      250),
    "time_shift":         (time_shift,         {"single", "multi"},                      150),
    "tonight_swap":       (tonight_swap,       {"single", "multi", "grid", "cancel"},    200),
    "flex_shift":         (flex_shift,         {"single", "multi", "grid"},              200),
    "cancel_misread":     (cancel_misread,     {"cancel"},                               150),
    "missing_constraint": (missing_constraint, {"multi", "grid"},                        150),
    "false_refuse":       (false_refuse,       {"single", "multi", "grid", "cancel"},    100),
    "intent_confusion":   (intent_confusion,   {"no_action", "off_topic"},                80),
}

# ---------- assembling pairs ----------
NOTE_TAG = "【操作員備註】"

def prompt_of(row):
    return [m for m in row["messages"] if m["role"] != "assistant"]

def gold_of(row):
    return next(m["content"] for m in row["messages"] if m["role"] == "assistant")

def with_note(prompt, new_note):
    """Same prompt, operator note replaced (used to strip the violation from refuse notes)."""
    p = [dict(m) for m in prompt]
    head = p[-1]["content"].split(NOTE_TAG)[0]
    p[-1]["content"] = head + NOTE_TAG + new_note
    return p

def make_pair(row, rejected_text, source):
    return {"id": row["id"], "category": row["category"], "source": source,
            "prompt": prompt_of(row),
            "chosen":   [{"role": "assistant", "content": gold_of(row)}],
            "rejected": [{"role": "assistant", "content": rejected_text}]}

def build_rule_pairs(rows, rng, caps, grade, per_prompt=2, rule_order=None):
    """Apply each rule to eligible rows; keep only rejected answers the grader marks wrong.
    Rarer rules go first so they get first pick of prompts; each prompt is used at most `per_prompt` times."""
    used, pairs = {}, []
    for name in (rule_order or list(RULES)):
        fn, cats, _ = RULES[name]
        cand = [r for r in rows if r["category"] in cats]
        rng.shuffle(cand)
        got = 0
        for r in cand:
            if got >= caps.get(name, 0): break
            if used.get(r["id"], 0) >= per_prompt: continue
            gold_s = gold_of(r)
            rej = fn(json.loads(gold_s), now_of(r["messages"][1]["content"]), rng)
            if rej is None: continue
            rej_s = json.dumps(rej, ensure_ascii=False)
            if grade(rej_s, gold_s)["correct"]: continue
            pairs.append(make_pair(r, rej_s, name)); used[r["id"]] = used.get(r["id"], 0) + 1; got += 1
    return pairs

def refuse_jobs(rows):
    """Split refuse rows into (mixed rows + stripped prompts for the model) and pure-violation pairs."""
    mixed, pure_pairs = [], []
    for r in rows:
        if r["category"] != "refuse": continue
        kind, v, rest = split_refuse(r["note"])
        if kind is None: continue
        if rest:
            mixed.append((r, with_note(prompt_of(r), rest)))
        else:
            rej = comply_with_violation(kind, v, now_of(r["messages"][1]["content"]))
            pure_pairs.append(make_pair(r, json.dumps(rej, ensure_ascii=False), "refuse_comply"))
    return mixed, pure_pairs

def refuse_partial_pairs(mixed, model_texts, grade, parse):
    """Model answered the note with the violation removed → that answer is what 'ignoring the violation' looks like."""
    pairs, dropped = [], 0
    for (r, _), text in zip(mixed, model_texts):
        obj = parse(text)
        if not obj or obj.get("intent") != "set_constraints" or not obj.get("constraints") \
                or grade(text, gold_of(r))["correct"]:
            dropped += 1; continue
        pairs.append(make_pair(r, text.strip(), "refuse_partial"))
    return pairs, dropped

def onpolicy_pairs(probe_records):
    """Wrong answers the SFT model actually produced in 01a (greedy or sampled)."""
    pairs, seen = [], set()
    for rec in probe_records:
        row = {"id": rec["id"], "category": rec["category"],
               "messages": rec["prompt"] + [{"role": "assistant", "content": rec["gold"]}]}
        for o in [rec["greedy"]] + rec["samples"]:
            if o["correct"] or (rec["id"], o["text"]) in seen: continue
            seen.add((rec["id"], o["text"]))
            pairs.append(make_pair(row, o["text"].strip(), "on_policy"))
    return pairs

print("規則載入完成:", list(RULES))

## 7. 步驟 1：模型真的寫錯過的答案（on_policy）

01a 每題有 1 個 greedy + 4 個抽樣回答，答錯的全部拿來當 rejected。同一題如果寫出一模一樣的錯答案，只留一筆。

In [ ]:
rng = random.Random(SEED)
pairs_onpolicy = onpolicy_pairs(probe)
print("on_policy 配對:", len(pairs_onpolicy))

## 8. 步驟 2：規則造錯（練習本 + 小考卷）

`build_rule_pairs` 對每一題套用規則，**評分函式說「錯」的才收**。小考卷用 validation，數量 = 練習本 × `EVAL_FRACTION`。

In [ ]:
pairs_rule_train = build_rule_pairs(train_rows, rng, CAPS, grade, per_prompt=PER_PROMPT, rule_order=RULE_ORDER)
eval_caps = {k: max(5, round(v * EVAL_FRACTION)) for k, v in CAPS.items()}
pairs_rule_eval = build_rule_pairs(val_rows, rng, eval_caps, grade, per_prompt=PER_PROMPT, rule_order=RULE_ORDER)
print("練習本:", dict(collections.Counter(p["source"] for p in pairs_rule_train)))
print("小考卷:", dict(collections.Counter(p["source"] for p in pairs_rule_eval)))

## 9. 步驟 3：refuse 題

`refuse_jobs` 把 refuse 題分成兩堆：

- **純違規**（例如「緊急推論停一小時應該沒差吧」）→ 直接用規則造「照做」的答案
- **混合題**（例如「RAG 更新往後挪三小時，另外這次不用人工審核」）→ 把違規那句拿掉，變成「RAG 更新往後挪三小時」，下一步交給模型回答

拿掉違規句的方法：你的 `generate.py` 只用兩種方式拼接（`{正常}，另外{違規}` 或 `{違規}。還有{正常}`），違規句子也只有固定幾種模板，所以可以精準拿掉。本機用 generate.py 產生的 3,954 題測試過，485 題 refuse 全部都能辨識。

In [ ]:
mixed_train, pure_train = refuse_jobs(train_rows)
mixed_eval,  pure_eval  = refuse_jobs(val_rows)
rng.shuffle(mixed_train); rng.shuffle(mixed_eval)
eval_partial_cap = max(10, round(REFUSE_PARTIAL_CAP * EVAL_FRACTION))
# 多送 20% 給模型：有些回答會不合格被丟掉，多送一點才湊得滿上限
mixed_train = mixed_train[: int(REFUSE_PARTIAL_CAP * 1.2)]
mixed_eval  = mixed_eval[: int(eval_partial_cap * 1.2)]
print(f"混合題（送模型）: train {len(mixed_train)} / val {len(mixed_eval)}　|　純違規: train {len(pure_train)} / val {len(pure_eval)}")
for r, p in mixed_train[:3]:
    print("\n原句:      ", r["note"])
    print("拿掉違規後:", p[-1]["content"].split(NOTE_TAG)[1])

## 10. 載入模型到 vLLM（跟 01a 完全一樣）

參數的意思請看 01a 第 9 格。這次只需要 greedy 回答大約 400 題，所以載入時間會比產生時間還長。

In [ ]:
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import time, traceback
from vllm import LLM, SamplingParams

t0 = time.time()
try:
    llm = LLM(
        model=MODEL_ID,
        dtype="half",
        max_model_len=2048,
        gpu_memory_utilization=0.85,
        tensor_parallel_size=1,
        enforce_eager=True,
        seed=SEED,
    )
except Exception:
    err = traceback.format_exc()
    with open(f"{OUT_DIR}/01a_vllm_error.txt", "w") as f:
        f.write(f"{VERSIONS}\n{GPU_INFO}\n\n{err}")
    print("VLLM_LOAD_FAILED\n" + err[-3000:])
    raise
LOAD_SECONDS = round(time.time() - t0)
print(f"✅ vLLM 載入完成，用時 {LOAD_SECONDS} 秒")

## 11. 讓模型回答「拿掉違規後」的備註

用 greedy（temperature=0），因為我們要的是模型**最可能**給出的答案，這樣的陷阱最真實。

`refuse_partial_pairs` 只收符合條件的回答：
- 是合法的 JSON
- `intent` 是 `set_constraints`，而且至少有一條限制（真的「照做了」）
- 跟原本的正確答案（refuse）比，評分函式判定為錯

不符合的就丟掉，例如模型連拿掉違規後的句子也拒絕了。

In [ ]:
import time
from vllm import SamplingParams
sp = SamplingParams(temperature=0, max_tokens=MAX_TOKENS)
jobs = mixed_train + mixed_eval
t0 = time.time()
outs = llm.chat([p for _, p in jobs], sp, use_tqdm=True)
texts = [o.outputs[0].text for o in outs]
print(f"產生 {len(texts)} 題，用時 {time.time()-t0:.0f} 秒")

pp_train, drop_t = refuse_partial_pairs(mixed_train, texts[:len(mixed_train)], grade, parse)
pp_eval,  drop_e = refuse_partial_pairs(mixed_eval,  texts[len(mixed_train):], grade, parse)
pp_train, pp_eval = pp_train[:REFUSE_PARTIAL_CAP], pp_eval[:eval_partial_cap]
print(f"refuse_partial: train {len(pp_train)}（丟掉 {drop_t}）/ val {len(pp_eval)}（丟掉 {drop_e}）")
for p in pp_train[:2]:
    print("\n備註:", p["prompt"][-1]["content"].split(NOTE_TAG)[1])
    print("CHOSEN:  ", p["chosen"][0]["content"][:200])
    print("REJECTED:", p["rejected"][0]["content"][:200])

## 12. 合併、最後檢查、統計

最後再檢查一次**每一筆**：
- chosen 一定要是正確答案（跟自己比 → 對）
- rejected 一定要是錯的（跟 chosen 比 → 錯）
- 同一題同一個陷阱答案不能重複

只要有一筆不符合，就直接報錯停下。練習本裡不能出現「兩個都對」或「兩個都錯」的配對。

In [ ]:
def finalize(pairs):
    seen, out = set(), []
    for p in pairs:
        key = (p["id"], p["rejected"][0]["content"])
        if key in seen: continue
        seen.add(key)
        c, r = p["chosen"][0]["content"], p["rejected"][0]["content"]
        assert grade(c, c)["correct"], ("chosen 不是正確答案", p["id"])
        assert not grade(r, c)["correct"], ("rejected 其實是對的", p["id"], p["source"])
        out.append(p)
    rng.shuffle(out)
    return out

train_pairs = finalize(pairs_onpolicy + pairs_rule_train + pure_train + pp_train)
eval_pairs  = finalize(pairs_rule_eval + pure_eval + pp_eval)
assert not ({p["id"] for p in train_pairs} & {p["id"] for p in eval_pairs}), "練習本和小考卷的題目重疊了"

import pandas as pd
stat = pd.DataFrame({"train": collections.Counter(p["source"] for p in train_pairs),
                     "eval":  collections.Counter(p["source"] for p in eval_pairs)}).fillna(0).astype(int)
stat.loc["TOTAL"] = stat.sum()
display(stat)
print("練習本各題型:", dict(collections.Counter(p["category"] for p in train_pairs)))

## 13. 每種陷阱看一個例子

**一定要親眼看過資料再拿去訓練。** 規則寫錯的話，數字上看不出來，打開來看才會發現。

In [ ]:
shown = set()
for p in train_pairs:
    if p["source"] in shown: continue
    shown.add(p["source"])
    print(f"\n========== {p['source']} ({p['category']}) ==========")
    print("備註:    ", p["prompt"][-1]["content"].split(NOTE_TAG)[1])
    print("CHOSEN:  ", p["chosen"][0]["content"])
    print("REJECTED:", p["rejected"][0]["content"])

## 14. 存檔 + 上傳到 Hugging Face

**格式**：TRL 的 `DPOTrainer` 直接吃這種「對話格式」：

```json
{"prompt":   [{"role": "system", ...}, {"role": "user", ...}],
 "chosen":   [{"role": "assistant", "content": "正確答案"}],
 "rejected": [{"role": "assistant", "content": "陷阱答案"}]}
```

另外保留 `id`、`category`、`source`，之後可以分析「哪一種陷阱學得最好」。

**上傳**：成功的話，02 可以直接 `load_dataset("yoyu001/grid-dispatch-dpo-pairs")`。如果 token 只有 read 權限，上傳會失敗，但檔案一樣存在 Output，02 改成 Add Input 讀取也可以。

In [ ]:
def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

save_jsonl(f"{OUT_DIR}/dpo_pairs_train.jsonl", train_pairs)
save_jsonl(f"{OUT_DIR}/dpo_pairs_eval.jsonl", eval_pairs)

pushed, push_error = False, ""
if PUSH_TO_HUB:
    try:
        from datasets import Dataset, DatasetDict
        DatasetDict({"train": Dataset.from_list(train_pairs), "test": Dataset.from_list(eval_pairs)}) \
            .push_to_hub(PAIRS_REPO, private=True)
        pushed = True
        print(f"✅ 已上傳到 https://huggingface.co/datasets/{PAIRS_REPO}（私人）")
    except Exception as e:
        push_error = repr(e)[:500]
        print("⚠️ 上傳失敗（多半是 token 只有 read 權限），檔案仍在 Output：", push_error)

summary = {
    "versions": VERSIONS, "gpu": GPU_INFO, "model": MODEL_ID, "seed": SEED,
    "caps": CAPS, "refuse_partial_cap": REFUSE_PARTIAL_CAP, "per_prompt": PER_PROMPT, "eval_fraction": EVAL_FRACTION,
    "counts": stat.reset_index().rename(columns={"index": "source"}).to_dict(orient="records"),
    "train_by_category": dict(collections.Counter(p["category"] for p in train_pairs)),
    "refuse_partial_dropped": {"train": drop_t, "eval": drop_e},
    "probe_file": hits[0] if hits else None,
    "pushed_to_hub": pushed, "push_error": push_error,
    "examples": {s: next({"note": p["prompt"][-1]["content"].split(NOTE_TAG)[1],
                          "chosen": p["chosen"][0]["content"], "rejected": p["rejected"][0]["content"]}
                         for p in train_pairs if p["source"] == s)
                 for s in sorted({p["source"] for p in train_pairs})},
}
with open(f"{OUT_DIR}/01b_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("✅ 完成。到 Version → Output 下載 01b_summary.json 傳給 Claude")